# Setup

In [17]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy pandas scikit-learn matplotlib mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [18]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [19]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

In [20]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)[:200]
            print(f'    └─ {kind}: {snippet}')

# Multiple agents: dataset broker -> clustering analyst

A useful multi-agent workflow is not just two prompts in sequence. The first agent should handle broad, messy context and reduce it to a small verified contract. The second agent should receive only that contract and perform a focused task.

In this notebook, the dataset broker can read a local dataset registry and inspect candidate CSV files. The clustering analyst never sees the registry, rejected datasets, or file-discovery logic. It only receives the broker's validated `DatasetContract`.

## Create a local dataset registry

In [21]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_iris, load_wine

DATA_DIR = Path('/content/data') if Path('/content').exists() else Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def save_sklearn_dataset(loader, path: Path, target_column: str) -> None:
    dataset = loader(as_frame=True)
    df = dataset.frame.copy()
    if 'target' in df.columns and target_column != 'target':
        df = df.rename(columns={'target': target_column})
    df.to_csv(path, index=False)


save_sklearn_dataset(load_breast_cancer, DATA_DIR / 'breast_cancer.csv', 'diagnosis')
save_sklearn_dataset(load_wine, DATA_DIR / 'wine.csv', 'wine_class')
save_sklearn_dataset(load_iris, DATA_DIR / 'iris.csv', 'species')

# Deliberately bad candidate: too small and no numeric feature columns.
pd.DataFrame({
    'sample_id': ['a', 'b', 'c'],
    'notes': ['text only', 'too small', 'not clusterable'],
}).to_csv(DATA_DIR / 'tiny_text_only.csv', index=False)

catalog = pd.DataFrame([
    {
        'name': 'breast_cancer',
        'domain': 'biomedicine',
        'path': str(DATA_DIR / 'breast_cancer.csv'),
        'target_column': 'diagnosis',
        'description': 'Tumor diagnostic measurements with 30 numeric features.',
    },
    {
        'name': 'wine',
        'domain': 'chemistry',
        'path': str(DATA_DIR / 'wine.csv'),
        'target_column': 'wine_class',
        'description': 'Chemical measurements of wines from three cultivars.',
    },
    {
        'name': 'iris',
        'domain': 'biology',
        'path': str(DATA_DIR / 'iris.csv'),
        'target_column': 'species',
        'description': 'Flower morphology dataset; useful but toy-sized.',
    },
    {
        'name': 'tiny_text_only',
        'domain': 'unknown',
        'path': str(DATA_DIR / 'tiny_text_only.csv'),
        'target_column': '',
        'description': 'Invalid candidate with only text columns and three rows.',
    },
])

catalog_path = DATA_DIR / 'dataset_catalog.csv'
catalog.to_csv(catalog_path, index=False)

## Agent 1: dataset broker

The broker has tools for reading the registry and inspecting CSV files.

In [22]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class DatasetContract(BaseModel):
    dataset_name: str | None = Field(description='Selected dataset name.')
    local_path: str | None = Field(description='Local path to the CSV file.')
    feature_columns: list[str] = Field(description='Numeric feature columns suitable for clustering.')
    label_column: str | None = Field(description='Optional label column to skip for clustering.')
    n_rows: int | None = Field(description='Number of rows in the selected CSV.')
    n_features: int | None = Field(description='Number of usable numeric features.')
    reason: str = Field(description='Short justification for the decision.')

dataset_broker = Agent(
    MODEL,
    output_type=DatasetContract,
    system_prompt=(
        'You are a dataset broker for a clustering workflow. Use the available tools. '
        'Read the catalog, inspect plausible CSV files, and return a concise DatasetContract. '
    ),
)

@dataset_broker.tool_plain
def read_dataset_catalog() -> list[dict]:
    return pd.read_csv(catalog_path).fillna('').to_dict(orient='records')

@dataset_broker.tool_plain
def inspect_csv(path: str, target_column: str | None = None) -> dict:
    csv_path = Path(path)
    if not csv_path.exists():
        return {'error': 'file not found'}

    df = pd.read_csv(csv_path)
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    feature_columns = [col for col in numeric_columns if col != target_column]

    return {
        'n_rows': len(df),
        'feature_columns': feature_columns,
        'label_column_ok': target_column in df.columns if target_column else False,
    }

In [23]:
selected_dataset = dataset_broker.run_sync(
    """Select a dataset for a biomedical clustering demonstration. must be a local CSV from the catalog must 
    have numeric feature columns suitable for clustering""",
    retries=2,
).output

selected_dataset

DatasetContract(dataset_name='breast_cancer', local_path='/content/data/breast_cancer.csv', feature_columns=['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension'], label_column='diagnosis', n_rows=569, n_features=30, reason="Breast cancer dataset is ideal for biomedical clustering: contains 569 tumor samples with 30 rich numeric features derived from diagnostic measurements (mean, error, and worst-case statistics). The target label 'diagnosis' can be used for evaluation. Thi

## Agent 2: clustering analyst

The clustering analyst receives only the validated `DatasetContract`. It has two tools: one to write a Python script and one to run it. The script runs with the same Python executable as this notebook, so it can import packages installed in the setup cell, including `pandas` and `scikit-learn`.

In [ ]:
from pydantic_ai import RunContext
from pydantic import field_validator
import subprocess
import sys

class ClusteringRun(BaseModel):
    script_path: str = Field(description='Path to the Python script that was written.')
    stdout: str = Field(description='Output from running the script.')
    returncode: int = Field(description='Process return code from running the script.')
    clusters_path: str = Field(description='Path to clusters.csv: the selected feature columns plus a "cluster" column (KMeans label per row).')
    explanation: str = Field(description='Brief explanation of the analysis.')

    @field_validator('clusters_path')
    @classmethod
    def check_clusters_exists(cls, path):
        if not Path(path).exists():
            raise ValueError(f'clusters_path does not exist: {path}')
        return path

clustering_analyst = Agent(
    MODEL,
    deps_type=DatasetContract,
    output_type=ClusteringRun,
    system_prompt=(
        'You are a clustering analyst. Write and run a complete Python script from the dataset contract. '
        'The script must load the CSV, select only the provided feature columns, standardize features, '
        'run KMeans, and compute silhouette_score. '
        'It must also save the selected feature columns together with a new "cluster" column '
        '(the KMeans label for each row) to "clusters.csv" in the same folder as the script '
    ),
)

@clustering_analyst.system_prompt
def add_dataset_contract(ctx: RunContext[DatasetContract]) -> str:
    contract = ctx.deps
    lines = [
        f'Dataset name: {contract.dataset_name}',
        f'CSV path: {contract.local_path}',
        f'Rows: {contract.n_rows}',
        f'Feature columns: {contract.feature_columns}',
        f'Label column: {contract.label_column}',
    ]
    return '\n'.join(lines)

@clustering_analyst.tool_plain
def write_python_script(filename: str, code: str) -> str:
    if not filename.endswith('.py'):
        filename = f'{filename}.py'
    script_path = DATA_DIR / filename
    script_path.write_text(code, encoding='utf-8')
    return str(script_path)

@clustering_analyst.tool_plain
def run_python_script(script_path: str) -> dict:
    result = subprocess.run(
        [sys.executable, script_path],
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        'returncode': result.returncode,
        'stdout': result.stdout,
        'stderr': result.stderr,
    }

clustering_result = clustering_analyst.run_sync(
    'Write and run the clustering analysis script for this dataset.',
    deps=selected_dataset,
    retries=2,
)

print(clustering_result.output.stdout)

In [25]:
pretty_print_trace(clustering_result)


[0] ModelRequest
    └─ SystemPromptPart: SystemPromptPart(content='You are a clustering analyst. Write and run a complete Python script from the dataset contract. The script must load the CSV, select only the provided feature columns, standa
    └─ SystemPromptPart: SystemPromptPart(content="Dataset name: breast_cancer\nCSV path: /content/data/breast_cancer.csv\nRows: 569\nFeature columns: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoot
    └─ UserPromptPart: UserPromptPart(content='Write and run the clustering analysis script for this dataset.', timestamp=datetime.datetime(2026, 7, 3, 9, 46, 54, 236903, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='write_python_script', args='{"filename": "clustering_analysis.py", "code": "import pandas as pd\\nimport numpy as np\\nfrom sklearn.preprocessing import StandardScaler\\nfrom s

[2] ModelRequest
    └─ ToolReturnPart: ToolReturnPart(tool_name='write_python_script'

## Exercise

Add a third agent, a **Reporter**, that turns the two upstream results into a human-readable report *and* a visualisation of the clusters.

Neither upstream agent can produce this alone: the broker never saw the clustering results, and the analyst never saw which datasets were rejected or why. The reporter is the only place both halves meet.

Keep the boundary honest:
- The reporter receives only the two upstream results — the `DatasetContract` and the `ClusteringRun` — bundled into one `deps` object.
- Its single tool is *additive*: it plots the clusters. It reads only the `clusters.csv` the analyst already saved (via `deps`), projects the feature columns to 2-D with PCA for display, and colours the points by the `cluster` column. Projecting for a picture is not re-doing the clustering — the labels come straight from the analyst.

The `plot_clusters` tool is **provided for you** below (it's just plotting boilerplate). Your job is the multi-agent wiring:

1. Design the `ClusterReport` output — the fields that tie the broker's *why* to the analyst's *results*.
2. Fill in `add_report_inputs` so both `ctx.deps.contract` and `ctx.deps.run` reach the model's prompt.

Fill in the `#TODO`s below, then run the last cell.

In [ ]:
from pydantic_ai import RunContext
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# The reporter depends on BOTH upstream results, so bundle them into one deps object.
class ReportInputs(BaseModel):
    contract: DatasetContract
    run: ClusteringRun

class ClusterReport(BaseModel):
    # TODO: design the report. A reasonable set of fields:
    #   dataset_choice: str       -- which dataset, and why the broker picked it
    #   n_clusters: int
    #   silhouette: float | None  -- from the analyst's run
    #   plot_path: str            -- PNG produced by plot_clusters
    #   recommendation: str       -- is the result trustworthy? what next?
    ...

reporter = Agent(
    MODEL,
    deps_type=ReportInputs,
    output_type=ClusterReport,
    system_prompt=(
        'You are a reporter. You are given a dataset-selection decision and a clustering run. '
        'Write a short report that connects the two, and call your plotting tool to visualise '
        'the clusters. You have no access to the raw data - use only what you are given.'
    ),
)

@reporter.system_prompt
def add_report_inputs(ctx: RunContext[ReportInputs]) -> str:
    # TODO: format ctx.deps.contract (the broker's choice + reason) and
    #       ctx.deps.run (the analyst's results) into the prompt.
    ...

# Provided for you: a narrow, additive tool. It reads ONLY the analyst's clusters.csv
# (via deps), projects the features to 2-D for display, and never touches the raw dataset.
# Registered with @reporter.tool (not @reporter.tool_plain) because it needs ctx.deps.
@reporter.tool
def plot_clusters(ctx: RunContext[ReportInputs]) -> str:
    df = pd.read_csv(ctx.deps.run.clusters_path)
    features = df.drop(columns='cluster')
    coords = PCA(n_components=2).fit_transform(features)

    fig, ax = plt.subplots()
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=df['cluster'], cmap='tab10', s=12)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'Clusters for {ctx.deps.contract.dataset_name}')
    ax.legend(*scatter.legend_elements(), title='cluster')

    plot_path = DATA_DIR / 'clusters_plot.png'
    fig.savefig(plot_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    return str(plot_path)

In [ ]:
# Run the reporter, passing both upstream results through deps:
#
# report = reporter.run_sync(
#     'Write the report and plot the clusters.',
#     deps=ReportInputs(contract=selected_dataset, run=clustering_result.output),
# ).output
# report